# Tutorial 4 — Author and inspect an IPF model

## Prerequisites

Complete Tutorials 1–3. This tutorial assumes you can declare primitive
components, package a `CompositePlan`, and consume a team-owned model.

## Objects introduced here

General `RLGC`, `load_q2d_rlgc()`, conductor-qualified transmission-line
pins, `interdigitated_capacitor`, shared-parameter fan-out, `AffineMap`,
`CoordinateRef`, `ReductionPipeline.ptc()`, Direct quantity selectors,
compiled schematics, and `run.explain()`.

## What the reader will build

A reusable synthetic intrinsic Purcell-filter model with four
one-conductor line sections, one two-conductor coupled section, an IDC
calibration, public pins, five public parameters, and an outer circuit
model.

## What inspection or Result is produced

This tutorial only builds and inspects declarations: the model default,
compiled schematic, and sealed request explanation. It does not
optimize, solve, select a winner, or request HB.

The repository remains a `CONVERGING` fail-fast scaffold, so the
executable cells document the candidate API but are deliberately not run
while rendering.

## Start with one general line carrier

The pain with separate “single” and “coupled” line APIs is that a
composite cannot use one wiring rule across its sections. `RLGC` instead
has an ordered conductor list: the readout and filter sections are
`N=1`; the shared coupled section is `N=2`. Its transmission-line pins
therefore require both the end (`head` or `tail`) and the conductor
name.

The model uses checked-in synthetic matrices. A frozen Q2D CSV can
replace a manual carrier without changing the composite topology or
conductor order. The repository does not invent a sample Q2D format
before the parser exists, so this is a replacement fragment rather than
an executable cell:

``` python
from scnsim import load_q2d_rlgc

coupled_rlgc = load_q2d_rlgc(
    "artifacts/coupled_q2d.csv",
    reference_conductor="ground",
    conductor_map={"Readout": "readout", "Filter": "filter"},
)
```

`load_q2d_rlgc()` is an alternative ingestion path, not an extractor
run. The source direction, matrix order, and provenance remain attached
to the imported carrier. An imported non-diagonal resistance matrix is
Direct-only; the synthetic manual `N=2` resistance matrix is diagonal so
Tutorial 5 can also request pump-off HB.

## Package the five sections once

The component author declares the multi-section `CompositePlan` once.
One `shared_short_length` `ParameterRef` fans out to both short
sections; the IDC receives three calibrated `AffineMap` values from one
finger-length ref. Each map’s `support` is its evidence interval, not a
global switch. The Composite exposes wiring pins (`feedline_in`,
`feedline_out`) separately from its analysis-only `CoordinateRef`
(`filter_open_tail`).

Show the reusable IPF Composite source

``` python
"""Public synthetic IPF Composite used to review the SCNSim V1 authoring UX.

The declarations intentionally contain no parser, compiler, solver, or hidden
success path.  They show how a custom Library author exposes five physical
parameters, calibrated affine mappings, external pins, and one internal
analysis coordinate while keeping the child topology private.
"""

from __future__ import annotations

from scnsim import (
    AffineMap,
    ComponentInstance,
    CompositePlan,
    Library,
    ParameterSpec,
    RLGC,
    library as sc,
    units as u,
)


class IPFLibrary(Library):
    """Immutable custom catalog for the synthetic intrinsic Purcell filter."""

    def intrinsic_purcell_filter(
        self,
        *,
        id: str,
        readout_rlgc: RLGC,
        filter_rlgc: RLGC,
        coupled_rlgc: RLGC,
    ) -> ComponentInstance:
        """Declare a five-section IPF with five public optimization parameters."""

        component = CompositePlan(id=id, library=self)

        readout_open_length = component.parameter(
            id="readout_open_length",
            baseline=2.4 * u.mm,
            spec=ParameterSpec(unit=u.mm),
        )
        shared_short_length = component.parameter(
            id="shared_short_length",
            baseline=0.9 * u.mm,
            spec=ParameterSpec(unit=u.mm),
        )
        coupled_length = component.parameter(
            id="coupled_length",
            baseline=1.6 * u.mm,
            spec=ParameterSpec(unit=u.mm),
        )
        filter_open_length = component.parameter(
            id="filter_open_length",
            baseline=2.1 * u.mm,
            spec=ParameterSpec(unit=u.mm),
        )
        idc_finger_length = component.parameter(
            id="idc_finger_length",
            baseline=52.0 * u.um,
            spec=ParameterSpec(unit=u.um),
        )

        readout_open = component.add(
            sc.transmission_line(
                id="readout_open",
                length=readout_open_length,
                rlgc=readout_rlgc,
                n_sections=24,
            )
        )
        readout_short = component.add(
            sc.transmission_line(
                id="readout_short",
                length=shared_short_length,
                rlgc=readout_rlgc,
                n_sections=12,
            )
        )
        coupled = component.add(
            sc.transmission_line(
                id="coupled",
                length=coupled_length,
                rlgc=coupled_rlgc,
                n_sections=20,
            )
        )
        filter_short = component.add(
            sc.transmission_line(
                id="filter_short",
                length=shared_short_length,
                rlgc=filter_rlgc,
                n_sections=12,
            )
        )
        filter_open = component.add(
            sc.transmission_line(
                id="filter_open",
                length=filter_open_length,
                rlgc=filter_rlgc,
                n_sections=24,
            )
        )

        calibration_support = (35.0 * u.um, 70.0 * u.um)
        idc = component.add(
            sc.interdigitated_capacitor(
                id="idc",
                terminal_1_to_reference_capacitance=AffineMap(
                    input=idc_finger_length,
                    slope=0.08 * u.fF / u.um,
                    intercept=1.2 * u.fF,
                    support=calibration_support,
                ),
                terminal_2_to_reference_capacitance=AffineMap(
                    input=idc_finger_length,
                    slope=0.09 * u.fF / u.um,
                    intercept=1.0 * u.fF,
                    support=calibration_support,
                ),
                terminal_mutual_capacitance=AffineMap(
                    input=idc_finger_length,
                    slope=0.12 * u.fF / u.um,
                    intercept=0.8 * u.fF,
                    support=calibration_support,
                ),
            )
        )

        feedline_in = component.net(
            readout_open.pin("head", conductor="readout"),
            idc.pin("terminal_1"),
        )
        component.net(
            readout_open.pin("tail", conductor="readout"),
            readout_short.pin("head", conductor="readout"),
        )
        component.net(
            readout_short.pin("tail", conductor="readout"),
            coupled.pin("head", conductor="readout"),
        )
        feedline_out = component.net(
            coupled.pin("tail", conductor="readout"),
        )

        filter_open_tail = component.net(
            filter_open.pin("tail", conductor="filter"),
        )
        component.net(
            filter_open.pin("head", conductor="filter"),
            coupled.pin("head", conductor="filter"),
        )
        component.net(
            coupled.pin("tail", conductor="filter"),
            filter_short.pin("head", conductor="filter"),
        )
        component.net(
            filter_short.pin("tail", conductor="filter"),
            idc.pin("terminal_2"),
        )

        component.expose_pin(id="feedline_in", at=feedline_in)
        component.expose_pin(id="feedline_out", at=feedline_out)
        component.expose_coordinate(
            id="filter_open_tail",
            at=filter_open_tail,
        )
        return component.build()


library = object.__new__(IPFLibrary)
"""Immutable synthetic custom Library object for the public example."""
```

That separation solves a common composition mistake: an outer
`CircuitPlan` can connect public pins, but cannot wire a
`CoordinateRef`. The latter exists only to retain or select an
observable later.

## Add the outer circuit and public Ports

The outer model attaches the IPF to coupling capacitors and the qubit,
then declares terminated feedline Ports and nonloading qubit-probe
Ports. The model keeps the resulting node, port, parameter, and
coordinate refs as its reusable public façade rather than asking each
notebook to rediscover IDs.

In [ ]:
from circuit_model import build_model

model = build_model()
model.shared_short_length.show()
model.idc_finger_length.show()

`parameter.show()` is the useful first inspection: it exposes the sealed
baseline, identity fan-out, affine calibration, and support without
compiling or solving.

## Derive views without changing topology

The qubit probe Ports are physical Ports but nonloading observations.
PTC compensates those evidenced probe shunts, then a pair transform
creates the qubit coordinates. From that same immutable lineage, the
model derives a terminal response view and a Direct-quantity view. The
first is port-realizable; the second retains the feedline and exposed
filter coordinate needed by roots, the transfer zero, residue-normalized
coupling, and linewidth selectors.

In [ ]:
from circuit_model import build_session

session = build_session(model, workspace="workspace/synthetic_ipf")
session.response_view
session.optimization_view

## Inspect the compiled physical circuit

The authoring schematic would keep the five logical line components. For
a structural audit, request the compiled representation: it expands
every pi section while preserving the declared `N=1`/`N=2` conductor
ordering and IDC attachment. This remains an inspection result, not a
numerical response.

In [ ]:
from scnsim import CircuitDiagramSpec

compiled = model.plan.render_schematic(
    CircuitDiagramSpec(
        representation="compiled",
        show_parameter_values=True,
    )
)
compiled.show()

## Inspect the model-owned recipe before any execution

The model, not the notebook consumer, owns the five-variable Direct
recipe. It combines two diagonal-root frequency selectors, a
transfer-zero selector, a residue-normalized coupling selector, and a
linewidth sum. Inspection gives the reader the bound quantities and
exact preflight request without invoking an optimizer.

In [ ]:
from circuit_model import IPFTarget
from scnsim import units as u

target = IPFTarget(
    readout_frequency=6.2 * u.GHz,
    filter_frequency=7.1 * u.GHz,
    transfer_zero_frequency=6.6 * u.GHz,
    coupling=45.0 * u.MHz,
    combined_linewidth=2.0 * u.MHz,
    response_frequencies=tuple((5.0 + 0.02 * i) * u.GHz for i in range(151)),
)
default_spec = model.build_default_optimization_spec(target)
default_spec.show()
default_spec.variable(model.idc_finger_length).bounds
session.run.explain(session.optimization_view, default_spec).show()

## Reusable model source

The model façade is the only reusable source for the manual RLGC
matrices, outer `CircuitPlan`, PTC lineage, typed views, and default
optimization recipe. It is included here rather than copied into the
tutorial so the rendered lesson and importable implementation cannot
drift.

Show the reusable IPF model source

``` python
"""Reusable public façade for the synthetic SCNSim IPF optimization example.

The model author owns topology, Ref lineages, quantity definitions, default
bounds, objective wiring, and optimizer controls.  A notebook consumer supplies
only a target and workspace unless they deliberately create an immutable custom
optimization spec.
"""

from __future__ import annotations

from dataclasses import dataclass
from os import PathLike

from circuit_library import library as custom
from scnsim import (
    CMAESSpec,
    CircuitPlan,
    CircuitRun,
    CostObjective,
    DiagonalRootSpec,
    CoordinateRef,
    ElectricNodeRef,
    NetworkViewRef,
    OptimizationSpec,
    OptimizationVariable,
    ParameterRef,
    PortRef,
    QuantitySum,
    RLGC,
    ReductionPipeline,
    ResidueNormalizedCouplingSpec,
    TransferZeroSpec,
    library as sc,
    units as u,
)


@dataclass(frozen=True)
class IPFTarget:
    """Consumer-owned synthetic targets; none is an acceptance Gate."""

    readout_frequency: object
    filter_frequency: object
    transfer_zero_frequency: object
    coupling: object
    combined_linewidth: object
    response_frequencies: object


@dataclass(frozen=True)
class IPFModel:
    """Reusable model-author surface: one Plan, public refs, and default recipe."""

    plan: CircuitPlan
    input_boundary: ElectricNodeRef
    output_boundary: ElectricNodeRef
    feedline_in: ElectricNodeRef
    feedline_out: ElectricNodeRef
    qubit_plus: ElectricNodeRef
    qubit_minus: ElectricNodeRef
    filter_open_tail: CoordinateRef
    feedline_in_port: PortRef
    probe_plus: PortRef
    probe_minus: PortRef
    readout_open_length: ParameterRef
    shared_short_length: ParameterRef
    coupled_length: ParameterRef
    filter_open_length: ParameterRef
    idc_finger_length: ParameterRef

    def quantity_specs(
        self,
    ) -> tuple[
        DiagonalRootSpec,
        DiagonalRootSpec,
        TransferZeroSpec,
        ResidueNormalizedCouplingSpec,
    ]:
        """Return the reusable Direct roots, zero, and coupling definitions."""

        readout_root = DiagonalRootSpec(
            coordinate=self.feedline_out,
            root_hint=6.0 * u.GHz,
        )
        filter_root = DiagonalRootSpec(
            coordinate=self.filter_open_tail,
            root_hint=7.0 * u.GHz,
        )
        transfer_zero = TransferZeroSpec(
            anchor=6.5 * u.GHz,
            family="Y",
            input_coordinate=self.feedline_out,
            output_coordinate=self.filter_open_tail,
        )
        coupling = ResidueNormalizedCouplingSpec(
            branch_a=readout_root,
            branch_b=filter_root,
            frequency=6.5 * u.GHz,
        )
        return readout_root, filter_root, transfer_zero, coupling

    def build_default_optimization_spec(self, target: IPFTarget) -> OptimizationSpec:
        """Build the model-author-owned five-variable default search recipe."""

        readout_root, filter_root, transfer_zero, coupling = self.quantity_specs()
        return OptimizationSpec(
            variables=(
                OptimizationVariable(
                    parameter=self.readout_open_length,
                    bounds=(1.8 * u.mm, 3.0 * u.mm),
                ),
                OptimizationVariable(
                    parameter=self.shared_short_length,
                    bounds=(0.6 * u.mm, 1.2 * u.mm),
                ),
                OptimizationVariable(
                    parameter=self.coupled_length,
                    bounds=(1.1 * u.mm, 2.1 * u.mm),
                ),
                OptimizationVariable(
                    parameter=self.filter_open_length,
                    bounds=(1.5 * u.mm, 2.8 * u.mm),
                ),
                OptimizationVariable(
                    parameter=self.idc_finger_length,
                    bounds=(40.0 * u.um, 68.0 * u.um),
                ),
            ),
            objectives=(
                CostObjective(
                    id="readout_frequency",
                    quantity=readout_root.frequency,
                    target=target.readout_frequency,
                    weight=100.0 * u.dimensionless,
                ),
                CostObjective(
                    id="filter_frequency",
                    quantity=filter_root.frequency,
                    target=target.filter_frequency,
                    weight=100.0 * u.dimensionless,
                ),
                CostObjective(
                    id="transfer_zero",
                    quantity=transfer_zero.frequency,
                    target=target.transfer_zero_frequency,
                    weight=30.0 * u.dimensionless,
                ),
                CostObjective(
                    id="residue_coupling",
                    quantity=coupling.magnitude,
                    target=target.coupling,
                    weight=10.0 * u.dimensionless,
                ),
                CostObjective(
                    id="combined_linewidth",
                    quantity=QuantitySum(
                        readout_root.linewidth,
                        filter_root.linewidth,
                    ),
                    target=target.combined_linewidth,
                    weight=5.0 * u.dimensionless,
                ),
            ),
            optimizer=CMAESSpec(seed=17, max_evaluations=400),
        )


@dataclass(frozen=True)
class IPFSession:
    """One sealed Run and its reusable response/optimization views."""

    run: CircuitRun
    response_view: NetworkViewRef
    optimization_view: NetworkViewRef


def _manual_rlgc() -> tuple[RLGC, RLGC, RLGC]:
    """Declare public synthetic one- and two-trace matrices."""

    readout = RLGC(
        conductors=("readout",),
        reference_conductor="ground",
        resistance_per_length=[[0.18]] * u.ohm / u.m,
        inductance_per_length=[[420.0]] * u.nH / u.m,
        conductance_per_length=[[0.0]] * u.S / u.m,
        capacitance_per_length=[[170.0]] * u.pF / u.m,
    )
    filter_line = RLGC(
        conductors=("filter",),
        reference_conductor="ground",
        resistance_per_length=[[0.22]] * u.ohm / u.m,
        inductance_per_length=[[395.0]] * u.nH / u.m,
        conductance_per_length=[[0.0]] * u.S / u.m,
        capacitance_per_length=[[162.0]] * u.pF / u.m,
    )
    coupled = RLGC(
        conductors=("readout", "filter"),
        reference_conductor="ground",
        resistance_per_length=[[0.18, 0.0], [0.0, 0.22]] * u.ohm / u.m,
        inductance_per_length=[[420.0, 75.0], [75.0, 395.0]] * u.nH / u.m,
        conductance_per_length=[[0.0, 0.0], [0.0, 0.0]] * u.S / u.m,
        capacitance_per_length=[[175.0, -22.0], [-22.0, 168.0]] * u.pF / u.m,
    )
    return readout, filter_line, coupled


def build_model() -> IPFModel:
    """Build the full synthetic Plan once for model-author inspection."""

    readout_rlgc, filter_rlgc, coupled_rlgc = _manual_rlgc()
    plan = CircuitPlan(id="synthetic_ipf")
    ipf = plan.add(
        custom.intrinsic_purcell_filter(
            id="ipf",
            readout_rlgc=readout_rlgc,
            filter_rlgc=filter_rlgc,
            coupled_rlgc=coupled_rlgc,
        )
    )
    input_cap = plan.add(sc.capacitor(id="input_cap", capacitance=12.0 * u.fF))
    output_cap = plan.add(sc.capacitor(id="output_cap", capacitance=12.0 * u.fF))
    qubit_coupler = plan.add(
        sc.capacitor(id="qubit_coupler", capacitance=4.0 * u.fF)
    )
    qubit = plan.add(
        sc.floating_parallel_single_junction_resonator(
            id="qubit",
            terminal_1_to_reference_capacitance=45.0 * u.fF,
            terminal_2_to_reference_capacitance=42.0 * u.fF,
            terminal_mutual_capacitance=16.0 * u.fF,
            josephson_inductance=420.0 * u.pH,
            junction_capacitance=2.0 * u.fF,
        )
    )

    input_boundary = plan.net(input_cap.pin("terminal_1"))
    feedline_in = plan.net(
        input_cap.pin("terminal_2"),
        ipf.pin("feedline_in"),
        id="feedline_in_node",
    )
    feedline_out = plan.net(
        ipf.pin("feedline_out"),
        output_cap.pin("terminal_1"),
        qubit_coupler.pin("terminal_2"),
        id="feedline_out_node",
    )
    output_boundary = plan.net(output_cap.pin("terminal_2"))
    qubit_plus = plan.net(
        qubit.pin("terminal_1"),
        qubit_coupler.pin("terminal_1"),
        id="qubit_plus",
    )
    qubit_minus = plan.net(qubit.pin("terminal_2"), id="qubit_minus")

    feedline_in_port = plan.add_port(
        id="feedline_in",
        at=input_boundary,
        role="terminated",
        reference_impedance=50.0 * u.ohm,
    )
    plan.add_port(
        id="feedline_out",
        at=output_boundary,
        role="terminated",
        reference_impedance=50.0 * u.ohm,
    )
    probe_plus = plan.add_port(
        id="qubit_probe_plus",
        at=qubit_plus,
        role="nonloading_probe",
        reference_impedance=50.0 * u.ohm,
    )
    probe_minus = plan.add_port(
        id="qubit_probe_minus",
        at=qubit_minus,
        role="nonloading_probe",
        reference_impedance=50.0 * u.ohm,
    )

    return IPFModel(
        plan=plan,
        input_boundary=input_boundary,
        output_boundary=output_boundary,
        feedline_in=feedline_in,
        feedline_out=feedline_out,
        qubit_plus=qubit_plus,
        qubit_minus=qubit_minus,
        filter_open_tail=ipf.coordinate("filter_open_tail"),
        feedline_in_port=feedline_in_port,
        probe_plus=probe_plus,
        probe_minus=probe_minus,
        readout_open_length=ipf.parameter("readout_open_length"),
        shared_short_length=ipf.parameter("shared_short_length"),
        coupled_length=ipf.parameter("coupled_length"),
        filter_open_length=ipf.parameter("filter_open_length"),
        idc_finger_length=ipf.parameter("idc_finger_length"),
    )


def build_session(
    model: IPFModel,
    *,
    workspace: str | PathLike[str],
) -> IPFSession:
    """Seal the Plan and derive the common PTC lineage and terminal views."""

    run = CircuitRun(plan=model.plan, workspace=workspace)
    transformed = run.original.reduce(
        ReductionPipeline()
        .ptc(model.probe_plus, model.probe_minus)
        .transform_pair(model.qubit_plus, model.qubit_minus, id="qubit")
    )
    response_view = transformed.reduce(
        ReductionPipeline().retain(model.input_boundary, model.output_boundary)
    )
    optimization_view = transformed.reduce(
        ReductionPipeline().retain(
            model.feedline_out,
            model.filter_open_tail,
        )
    )
    return IPFSession(
        run=run,
        response_view=response_view,
        optimization_view=optimization_view,
    )
```

## Learned objects and next

You built and inspected `RLGC`, a multi-section `CompositePlan`, public
pins and `CoordinateRef`, outer Ports, PTC-derived views, Direct
selectors, and the model-owned default recipe. Next, Tutorial 5 executes
that sealed recipe, reuses its exact winner, and resolves the recorded
request after restart.